# Day 12(M3 Day03) 실습 — pgvector·RAG로 내 문서 기반 답변 만들기

**목표**: 문서를 청킹해 pgvector에 저장·검색하고, 리트리버(Retriever)+프롬프트+LLM을 체인으로 묶어 근거 기반으로 답한다.
**구성**: Part 1 청킹·저장·검색(+청킹/연결 오류 2종) → Part 2 청크 크기 실험(+사내 복리후생 문서 도메인 반복) → Part 3 RAG 체인(+Day01~02 면접 코치를 pgvector로 업그레이드) → Part 4 PDF 문서로 RAG(불러오기·잡음 정리·쪽 번호 근거·메타데이터 필터)

> **참고:** 실습 전 가상환경 활성화, `.env`의 `OPENAI_API_KEY`·`DATABASE_URL`을 확인한다. `uv add langchain-postgres langchain-text-splitters pymupdf`와 pgvector 확장(`CREATE EXTENSION vector`)이 필요하다(이미 준비돼 있다면 생략). Part 4는 `data/` 폴더에 PDF 3종이 있어야 한다(배포 문서 「실습용 PDF 내려받기」).

## 0. 환경 준비 (연결·모델)

In [1]:
import os
os.environ["LANGSMITH_TRACING"] = "false"
os.environ["LANGCHAIN_TRACING_V2"] = "false"

from langchain_postgres import PGVector
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain_text_splitters import RecursiveCharacterTextSplitter
from dotenv import load_dotenv

load_dotenv()
PGVECTOR_URL = os.getenv("DATABASE_URL").replace("postgresql://", "postgresql+psycopg://")

print("준비 완료")

준비 완료


In [2]:
from sqlalchemy import create_engine
pg_engine = create_engine(PGVECTOR_URL, pool_size=2, max_overflow=0)

In [3]:
embeddings = OpenAIEmbeddings(model="text-embedding-3-small")
llm = ChatOpenAI(model="gpt-4o-mini")

## Part 1. 청킹·저장·검색 — 기초

완성 코드를 직접 쳐서 문서를 자르고 pgvector에 저장한 뒤, 리트리버로 관련 조각을 찾아본다.

> **참고:** M1 Day01에서 프롬프트에 넣었던 역할·지시문·맥락(`context`)·예시는 전부 **정적 콘텐츠**였다 — 미리 정해서 프롬프트에 고정으로 박아둔 것이다. 오늘부터는 그 `context`를 하드코딩하는 대신, 질문마다 **그때그때 검색해서 찾아온 문서**로 채운다 — 이게 **동적 콘텐츠**이고, RAG(Retrieval-Augmented Generation)의 핵심이다.

### 1-1. 문서 준비·청킹

긴 문서를 검색하기 좋은 크기로 자른다.

In [4]:
# 문서: 무장, 문단, 표, 이미지, 다양한 요소를 포함

DOC = """LangChain은 LLM 애플리케이션을 만드는 프레임워크다.
LCEL은 파이프 기호로 부품을 연결하는 방식이다.
임베딩은 문장을 숫자 벡터로 바꾼다.
pgvector는 PostgreSQL에서 벡터를 저장하고 검색하는 확장이다.
RAG는 관련 문서를 찾아 근거로 답하는 방식이다."""

# TODO: chunk_size=50, chunk_overlap=0인 RecursiveCharacterTextSplitter로 DOC을 잘라 chunks에 저장하세요
splitter = RecursiveCharacterTextSplitter(chunk_size=50, chunk_overlap=0)
chunks = splitter.split_text(DOC)
len(chunks)

4

In [5]:
type(chunks)

list

In [6]:
for c in chunks:
    print(c)
    print('-' * 50)

LangChain은 LLM 애플리케이션을 만드는 프레임워크다.
--------------------------------------------------
LCEL은 파이프 기호로 부품을 연결하는 방식이다.
임베딩은 문장을 숫자 벡터로 바꾼다.
--------------------------------------------------
pgvector는 PostgreSQL에서 벡터를 저장하고 검색하는 확장이다.
--------------------------------------------------
RAG는 관련 문서를 찾아 근거로 답하는 방식이다.
--------------------------------------------------


### 1-2. pgvector에 저장

PGVector가 임베딩·저장을 한 번에 처리한다.

In [8]:
# TODO: embeddings·collection_name("my_docs")·connection(engine)으로 PGVector store를 만들고
#       (pre_delete_collection=True, use_jsonb=True) chunks를 add_texts로 저장하세요

store = PGVector(
    embeddings=embeddings, # embedding 모델 객체
    collection_name="my_docs", #저장소 이름
    connection=pg_engine, # pgvector connection
    use_jsonb=True,
    pre_delete_collection=True #실습중 켜기
)

store.add_texts(chunks)

print("저장 완료")

저장 완료


### 1-3. 유사도 검색

질문과 가까운 조각을 찾는다.

In [9]:
# TODO: store.similarity_search로 "pgvector가 뭐야?"와 가까운 조각 2개를 hits에 찾으세요

query = "pgvector가 뭐야?"

hits = store.similarity_search(query, k=1)
for h in hits:
    print("결과: ", h.page_content)

결과:  pgvector는 PostgreSQL에서 벡터를 저장하고 검색하는 확장이다.


In [10]:
hits = store.similarity_search_with_score(query, k=4)
for h, score in hits:
    print("결과: ", h.page_content, score) # score는 거리값, 가까우면 순서가 높다.

결과:  pgvector는 PostgreSQL에서 벡터를 저장하고 검색하는 확장이다. 0.340777698245945
결과:  LCEL은 파이프 기호로 부품을 연결하는 방식이다.
임베딩은 문장을 숫자 벡터로 바꾼다. 0.775669009972781
결과:  RAG는 관련 문서를 찾아 근거로 답하는 방식이다. 0.807979745211868
결과:  LangChain은 LLM 애플리케이션을 만드는 프레임워크다. 0.811061178305428


### 1-4. 리트리버(retriever)로 변환

체인에 꽂을 수 있는 리트리버로 바꾼다.
기본체인: prompt | llm | parser -> Runnable

In [ ]:
# TODO: store.as_retriever(search_keywords={"k": 2})로 retriever를 만들고
#       "임베딩이 뭐야?"를 invoke해 docs에 저장하세요
# hits = store.similarity_search(query, k=1)

retriever = store.as_retriever(search_keywords={"k": 2})
ret_result = retriever.invoke("임베딩이 뭐야?")
ret_result

[Document(id='5e2e4760-7a5b-43dc-a159-2a1ca35c1e5b', metadata={}, page_content='LCEL은 파이프 기호로 부품을 연결하는 방식이다.\n임베딩은 문장을 숫자 벡터로 바꾼다.'),
 Document(id='57254241-b1fe-4759-b181-78676b3b502f', metadata={}, page_content='LangChain은 LLM 애플리케이션을 만드는 프레임워크다.'),
 Document(id='29b9ec5d-9e33-4d64-be49-df93bd9009b5', metadata={}, page_content='pgvector는 PostgreSQL에서 벡터를 저장하고 검색하는 확장이다.'),
 Document(id='7b09f857-969c-4fad-a2f0-d4cb5cc65af7', metadata={}, page_content='RAG는 관련 문서를 찾아 근거로 답하는 방식이다.')]

### 1-5. 오류 대응 ① — chunk_overlap이 chunk_size보다 크면?

겹침(overlap)을 자르는 크기보다 크게 주면 어떻게 되는지 확인한다.

In [25]:
# TODO: chunk_size=50, chunk_overlap=60인 RecursiveCharacterTextSplitter를 만들어 DOC을 잘라보세요
splitter = RecursiveCharacterTextSplitter(chunk_size=50, chunk_overlap=10, separators=[""])
chunks = splitter.split_text(DOC)

for c in chunks:
    print(c)
    print('-'*50)

LangChain은 LLM 애플리케이션을 만드는 프레임워크다.
LCEL은 파이프 기호로 부
--------------------------------------------------
파이프 기호로 부품을 연결하는 방식이다.
임베딩은 문장을 숫자 벡터로 바꾼다.
pgvec
--------------------------------------------------
바꾼다.
pgvector는 PostgreSQL에서 벡터를 저장하고 검색하는 확장이다.
RA
--------------------------------------------------
는 확장이다.
RAG는 관련 문서를 찾아 근거로 답하는 방식이다.
--------------------------------------------------


> **주의:** 결과는 실제 실행에서 직접 확인한다. `chunk_overlap`은 반드시 `chunk_size`보다 작아야 한다 — 겹치는 부분이 조각 자체보다 클 수는 없기 때문이다.

### 1-6. 실측 확인 — 연결 스킴을 안 바꾸면?

`DATABASE_URL`을 `postgresql+psycopg://`로 바꾸지 않고 그대로 PGVector에 넘기면 어떻게 되는지 직접 실행해서 확인한다.

In [ ]:

    # TODO: connection에 os.getenv("DATABASE_URL")을 스킴 변환 없이 그대로 넣어 PGVector를 만들고
    #       ["테스트 문장"]을 add_texts로 저장해보세요




> **참고:** 실제로 실행하면(이 환경·버전 기준) 스킴을 안 바꿔도 **오류 없이 저장·검색까지 정상 동작한다** — `langchain-postgres`가 내부적으로 스킴을 알아서 처리하기 때문으로 보인다. 문서(1-1 셋업 셀)가 안내하는 `postgresql+psycopg://` 변환이 '반드시 필요한' 것은 아닐 수 있다는 뜻이다. 다만 이는 라이브러리 버전에 따라 달라질 수 있는 동작이므로, 공식적으로 안내된 스킴 변환은 그대로 지키는 것이 안전하다 — **직접 실행해서 확인하기 전까지는 문서의 '반드시 이렇게 해야 한다'는 설명도 가정일 뿐**이라는 점이 이번 확인의 핵심이다.

### 관찰 정리

- 저장한 조각을 질문으로 어떻게 찾아 오나?
- 1-5의 오류는 어떤 값을 잘못 넣었을 때 생겼나?
- 1-6에서 확인한 것처럼, 문서의 '반드시 이렇게 해야 한다'는 설명도 실제로 돌려 보기 전까지는 가정일 뿐이라는 원칙은 왜 중요한가?

## Part 2. 청크 크기와 검색 품질 — 응용

자르는 크기(chunk_size)에 따라 검색 결과가 어떻게 달라지는지 관찰한다.

### 2-1. 긴 문서와 저장 함수

크기·컬렉션 이름을 받아 저장하는 함수를 만든다.

In [ ]:
LONG = ("파이썬은 1991년 귀도 반 로섬이 발표한 프로그래밍 언어다. "
        "문법이 간결해 배우기 쉽다. "
        "데이터 분석과 인공지능 분야에서 널리 쓰인다. "
        "라이브러리 생태계가 방대하다. "
        "인터프리터 언어라 실행 전 컴파일이 필요 없다.")

# TODO: size(chunk_size)·name(collection_name)을 받아 LONG을 잘라 PGVector에 저장하고
#       (store, chunks)를 반환하는 build 함수를 만드세요

def build(size, name):
    splitter = RecursiveCharacterTextSplitter(chunk_size=size, chunk_overlap=0)
    chunks = splitter.split_text(LONG)

    s = PGVector(
        embeddings=embeddings,
        collection_name=name,
        connection=pg_engine,
        use_jsonb=True,
        pre_delete_collection=True
    ) # 주소
    s.add_texts(chunks)
    return s, chunks

In [39]:
s_test, sc = build(30, "small_test")


In [40]:
l_test, lc = build(100, "long_test")

In [41]:
q = "파이썬은 언제 만들어졌나?"
s_result = s_test.similarity_search(q, k=1)
l_result = l_test.similarity_search(q, k=1)

In [ ]:
for h in s_result: # 30
    print(h.page_content)

파이썬은 1991년 귀도 반 로섬이 발표한 프로그래밍


In [ ]:
for h in l_result: # 100
    print(h.page_content)

파이썬은 1991년 귀도 반 로섬이 발표한 프로그래밍 언어다. 문법이 간결해 배우기 쉽다. 데이터 분석과 인공지능 분야에서 널리 쓰인다. 라이브러리 생태계가 방대하다. 인터프리터


### 2-2. 작은 청크(30) vs 큰 청크(120)

같은 문서를 서로 다른 크기로 저장한다.

In [ ]:
# TODO: build(30, "small_chunks")와 build(120, "large_chunks")로 small, sc / large, lc를 만드세요



### 2-3. 같은 질문으로 검색 비교

In [ ]:
q = "파이썬은 언제 만들어졌나?"


### 관찰

| 크기 | 조각 수 | 장점 | 단점 |
| --- | --- | --- | --- |
| 작게(30) | 여러 개 | 정확히 겨냥 | 앞뒤 문맥 부족 |
| 크게(120) | 적음 | 문맥 풍부 | 군더더기 섞임 |

### 2-4. 다른 도메인 반복 — 사내 복리후생 안내문

같은 방법을 다른 도메인에도 적용해본다. 이번엔 사내 복리후생 안내문으로 청크 크기 비교를 반복한다.

In [13]:
BENEFIT = ("우리 회사는 전 직원에게 연 100만원의 자기계발비를 지원한다. "
           "도서 구입, 온라인 강의, 자격증 시험 응시료에 사용할 수 있다. "
           "매년 1월에 사용 한도가 초기화되며 이월되지 않는다. "
           "신청은 사내 포털에서 영수증을 첨부해 진행한다. "
           "승인까지 보통 3영업일이 걸린다.")

# TODO: build_benefit(30, "benefit_small")와 build_benefit(120, "benefit_large")로
#       b_small, bsc / b_large, blc를 만드세요

def build_benefit(size, name):
    splitter = RecursiveCharacterTextSplitter(chunk_size=size, chunk_overlap=0)
    chunks = splitter.split_text(BENEFIT)

    s = PGVector(
        embeddings=embeddings,
        collection_name=name,
        connection=pg_engine,
        use_jsonb=True,
        pre_delete_collection=True
    ) # 주소
    s.add_texts(chunks)
    return s, chunks


In [14]:
b_small, bsc = build_benefit(30, "benefit_small")
b_large, blc = build_benefit(120, "benefit_large")

Collection not found
Collection not found


In [18]:
q = "자기개발비 신청 가능하니?"
bs_result = b_small.similarity_search(q, k=1)
bl_result = b_large.similarity_search(q, k=1)

In [21]:
for b in bs_result: # 30
    print(b.page_content)

우리 회사는 전 직원에게 연 100만원의 자기계발비를


In [22]:
for b in bl_result: # 100
    print(b.page_content)

우리 회사는 전 직원에게 연 100만원의 자기계발비를 지원한다. 도서 구입, 온라인 강의, 자격증 시험 응시료에 사용할 수 있다. 매년 1월에 사용 한도가 초기화되며 이월되지 않는다. 신청은 사내 포털에서 영수증을


### 관찰 정리

- 파이썬 도메인과 복리후생 도메인 모두에서, 작은 청크·큰 청크의 장단점이 그대로 나타났는가?
- 실무 문서라면 어떤 크기가 더 적당할지 어떻게 정할 수 있을까?

## Part 3. 미니 프로젝트 — RAG 체인

리트리버 + 프롬프트 + LLM을 체인으로 묶어 문서 기반으로 답한다.

### 3-1. 문서 인덱싱

자르기 → 저장 → 리트리버 변환을 한 번에 해 둔다.

In [28]:
DOC = """LangChain은 LLM 애플리케이션을 만드는 프레임워크다.
LCEL은 파이프 기호로 부품을 연결하는 방식이다.
임베딩은 문장을 숫자 벡터로 바꾼다.
pgvector는 PostgreSQL에서 벡터를 저장하고 검색하는 확장이다.
RAG는 관련 문서를 찾아 근거로 답하는 방식이다."""

# TODO: DOC을 청킹해 chunks를 만들고, PGVector store(collection_name="rag_docs")에 저장한 뒤
#       search_kwargs={"k": 2}인 retriever로 변환하세요

chunks = RecursiveCharacterTextSplitter(chunk_size=50, chunk_overlap=0).split_text(DOC)

store = PGVector(
        embeddings=embeddings,
        collection_name="rag_docs",
        connection=pg_engine,
        use_jsonb=True,
        pre_delete_collection=True
    ) # 주소

store.add_texts(chunks)

retriever = store.as_retriever(search_kwargs={"k": 2})

print("인덱싱 완료")

Collection not found


인덱싱 완료


### 3-2. RAG 체인 구성

검색된 조각을 `{context}`로, 질문을 `{question}`으로 넣는다.

In [30]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

# TODO: '근거에 없으면 모른다고 답해라'는 지시를 담은 prompt를 ChatPromptTemplate.from_template로 만드세요
#       ({context}, {question} 변수 사용)

prompt = ChatPromptTemplate.from_template(
    '다음에 나오는 근거만 사용해서 답하시오. 근거에 없으면 모른다고 답하시고 \n\n'
    '근거 : {context} \n\n 질문 : {question}'
)

# TODO: retriever | format_docs를 context로, RunnablePassthrough()를 question으로 하는
#       chain을 prompt | llm | StrOutputParser()까지 연결해 만드세요

def format_docs(doc):
    return "\n".join( d.page_content for d in doc )

chain = (
    {
        "context": retriever | format_docs,
        "question": RunnablePassthrough()
    }
    ) | prompt | llm | StrOutputParser()

print("체인 준비 완료")

체인 준비 완료


In [31]:
chain.invoke('pgvector가 뭐야')

'pgvector는 PostgreSQL에서 벡터를 저장하고 검색하는 확장입니다.'

In [32]:
chain.invoke('파이썬에 대해 3줄로 답변해줘')

'모른다고 답하겠습니다.'

### 3-3. 질문·환각 확인

문서에 있는 질문은 답하고, 없는 질문은 모른다고 해야 정상이다.

In [ ]:

# TODO: 문서에 없는 질문("파이썬 창시자가 누구야?")을 chain에 invoke해보세요

### 3-4. 근거 확인

답변이 어떤 조각을 근거로 했는지 함께 확인한다.

In [33]:
# 근거 확인: 어떤 조각을 참고했나
q = '레그가 뭐야?'
chain.invoke(q)

'RAG는 관련 문서를 찾아 근거로 답하는 방식이다.'

In [34]:
docs = retriever.invoke(q)
for d in docs:
    print(d.page_content)

RAG는 관련 문서를 찾아 근거로 답하는 방식이다.
LangChain은 LLM 애플리케이션을 만드는 프레임워크다.


### 관찰

| 질문 | 문서에 있나 | 결과 |
| --- | --- | --- |
| pgvector가 뭐야? | 있음 | 근거로 답함 |
| RAG가 뭐야? | 있음 | 근거로 답함 + 출처 확인 |
| 파이썬 창시자? | 없음 | 모른다 |

## Part 3-확장. Day01~02 통합 — 면접 코치를 pgvector로 업그레이드

Day01(M3)에서는 채용 공고를 파이썬 리스트+수동 코사인 유사도로 검색했다. 오늘은 같은 채용 공고를 **pgvector에 저장**하고 `retriever`로 검색하도록 바꾼다. Day02(M3)의 세션 저장(`save`/`load`)과 M1 Day04의 방어 로직은 그대로 재사용한다.

### 세션 저장소 연결 (Day02 재사용)

In [ ]:
# TODO: Day02(M3)의 psycopg2 연결과 save·load 함수를 그대로 옮겨오세요
#       (session_messages 테이블은 Day02에서 이미 만들어져 있다.
#        1과목 챗봇의 messages 테이블과 다른 테이블이므로 이름을 바꾸지 않는다)

### 채용 공고를 pgvector에 저장 (Day01 업그레이드)

In [ ]:
job_postings = [
    "백엔드 개발자: Python/Java 기반 서버 개발, RESTful API 설계, DB 최적화 경험 우대",
    "프론트엔드 개발자: React/TypeScript, 반응형 UI 구현, 웹 접근성 이해 우대",
    "데이터 분석가: SQL/Python 데이터 분석, 대시보드 제작, 통계적 사고 우대",
    "마케터: 콘텐츠 기획, 데이터 기반 캠페인 분석, SNS 채널 운영 경험 우대",
]
# TODO: job_postings를 collection_name="job_postings"인 PGVector store에 저장하고
#       (pre_delete_collection=True) search_kwargs={"k": 1}인 job_retriever로 변환하세요






print("채용 공고 pgvector 저장 완료")

### 방어 로직 (M1 Day04 재사용)

인젝션 방어는 M1 Day04에서 만든 것을 그대로 가져온다.

In [ ]:
# TODO: M1 Day04의 위험 문구·금지어 목록과 input_guard·output_guard를 옮겨오세요






### pgvector 기반 세션 면접 코치

Day02(M3)의 `session_coach`와 구조는 같지만, 채용 공고 검색을 수동 코사인 유사도 대신 `job_retriever`(pgvector)로 바꾼다. Day02에서 배운 대로, 검색 결과는 참고 자료일 뿐이라는 지시는 그대로 유지한다.

In [ ]:

    # TODO: job_retriever로 message와 관련된 채용 공고를 related_docs에서 찾아 related_job을 만들고,
    #       (참고 자료일 뿐이라는 지시를 포함한) SystemMessage + 대화 기록으로 msgs를 만든 뒤
    #       llm.invoke(msgs).content로 answer를 만드세요




### 최종 테스트

In [ ]:
# 다시 실행할 때 이전 실행의 대화가 섞이지 않도록 테스트 세션을 비운다

# TODO: 인젝션을 시도하는 질문을 넣어보세요

## Part 4. PDF 문서로 RAG — 쪽 번호까지 근거 제시

지금까지는 코드에 적은 짧은 문장을 문서로 썼다. 실제 보고서 PDF 3종으로 같은 과정을 밟는다. 새로 생기는 일은 세 가지다.

| 새로 생기는 일 | 절 |
| --- | --- |
| 파일에서 텍스트를 꺼낸다 | 4-1 |
| 쪽마다 반복되는 머리말·쪽 번호를 지운다 | 4-2 |
| 어느 파일 몇 쪽인지(메타데이터)를 답변까지 들고 간다 | 4-3 ~ 4-6 |

> **참고:** PDF는 배포 문서 「실습용 PDF 내려받기」대로 `data/` 폴더에 받아 둔다. NIPA 이용 규칙상 제3자 배포가 금지되어 저장소에는 PDF가 없다.

In [64]:
PDF_FILES = ['blockchain.pdf', 'ict_japan.pdf', 'ict_usa.pdf']

### 4-1. PDF 불러오기 — 쪽마다 Document 하나

`pymupdf`로 PDF를 열어 쪽마다 텍스트를 꺼내고, 파일명과 쪽 번호를 `metadata`에 담는다. `Document`는 LangChain에서 문서 조각을 담는 그릇이다(`page_content` + `metadata`).

In [65]:
from pathlib import Path

import pymupdf
from langchain_core.documents import Document

# 노트북을 프로젝트 루트 또는 notebooks 폴더에서 실행해도 data를 찾는다.
data_candidates = [Path("data"), Path("notebooks/data")]
DATA_DIR = next((path for path in data_candidates if path.is_dir()), None)
if DATA_DIR is None:
    raise FileNotFoundError("data 폴더를 찾을 수 없습니다. PDF를 data/ 폴더에 넣어 주세요.")

pdf_files = sorted(DATA_DIR.glob("*.pdf"))
if not pdf_files:
    raise FileNotFoundError(f"{DATA_DIR.resolve()}에 PDF 파일이 없습니다.")

def load_pdf(file_path: str | Path) -> list[Document]:
    pdf_path = Path(file_path)
    if not pdf_path.is_file():
        raise FileNotFoundError(f"PDF 파일을 찾을 수 없습니다: {pdf_path}")

    documents = []
    with pymupdf.open(pdf_path) as pdf:
        for page_number, page in enumerate(pdf, start=1):
            documents.append(
                Document(
                    page_content=page.get_text(),
                    metadata={"source": pdf_path.name, "page": page_number},
                )
            )
    return documents

In [66]:
pages = load_pdf('data/ict_japan.pdf')
len(pages)

32

In [67]:
pages[1].page_content

'Ⅰ ICT 국가 산업 현황 \n 4\n(*) SUMMARY\n1. 국가 개황\n2. ICT 정부 기관\n3. ICT 주요 정책\n4. ICT 주요 법령 및 규제\n5. ICT 주요 기업\n6. 한국 협력 및 국내기업 진출 사례\nⅡ ICT 주요 동향\n 17\n(*) SUMMARY\n1. 인공지능(AI)\n2. 스마트팩토리\n3. 디지털 콘텐츠\n4. 헬스케어\n5. 핀테크\n \nⅢ 스타트업 생태계\n 24\n(*) SUMMARY\n1. 사카나 AI(Sakana AI)\n2. 고(Go)\n3. 오픈(Opn)\n4. 텔레이그지스턴스(Telexistence)\n5. 무진(Mujin)\n※ 참고 문헌\n'

In [68]:
pages[5].page_content

'Ⅰ ICT 국가 산업 현황 \n6\n2. ICT 정부 기관\n① 총무성\n표 | 일본 총무성(MIC)\n출처 : 일본 총무성\n구분\n내용\n기관명\n총무성\n(総務省, Ministry of Internal Affairs and Communications)\n설립 연도\n2001년\n주요 \n산하 기관\n정보통신국(情報通信国), 국립정보통신기술연구소(NICT), \n사이버보안총괄관(サイバーセキュリティ統括官) 등\n홈페이지\nwww.soumu.go.jp\n주요 인사\n§\n무라카미 세이이치로 (村上\u3000誠一郎)\n§\n일본 총무성 총무대신\n§\n2024년 10월 총무대신 취임(석파내각)\n§\n2024년 11월 연임(제2차 석파내각)\n조직도\nn 국가 기본 구조와 국민 생활 기반에 관한 핵심 행정 기능 수행\n- 행정 운영 개선, 지방 행정, 선거, 소방 방재 등 국가 기반 행정 업무 총괄\n- 통신 및 우편 서비스 등 국민 생활 관련 서비스 제공 및 관리 담당\n- ICT 정책의 수립 및 실행, 사이버보안 정책 추진\nn ICT 기반 구축 및 국제 협력을 통한 디지털 혁신 가속화\n- 2024년 6월, 5G 및 차세대 통신 기술 발전 및 네트워크 환경 고도화를 위한 \n주파수 재편 계획 발표\n- 2024년 8월, 사회 전반에 디지털 기술 도입을 확산시키기 위한 디지털 전환 \n(DX) 촉진 전략 수립. 이 전략에는 행정 서비스의 디지털화, 디지털 인프라 \n강화, 디지털 인재 양성 등이 포함\n- 2024년 11월, 정부 및 공공기관의 보안 체계 강화를 위해 사이버 공격 대응 능력 \n향상, 정보 시스템 보호 조치 확대 등을 포함하는 사이버보안 강화 방안 마련\n'

> **참고:** LangChain에도 같은 일을 하는 `PyMuPDFLoader`가 있다. 다만 이 로더가 들어 있는 `langchain-community` 패키지는 지원 종료 예정이라 불러올 때 경고가 나온다. 로더가 하는 일은 위 함수가 전부이므로 직접 만들어 쓴다. `PyMuPDFLoader`의 `page`는 0부터 센다는 점도 다르다.

### 4-2. 쪽마다 반복되는 잡음 정리

4-1 출력의 앞부분을 보면 본문 앞에 `국가별 ICT 시장동향`(머리말)과 `7`(쪽 번호)이 붙어 있다. 모든 쪽에 반복되므로 검색에 방해만 된다. 또 양쪽 정렬 때문에 문장 중간에 줄바꿈(`\n`)이 들어가 있다.

In [69]:
# TODO: HEADER에 맞는 부분을 지우고, 줄바꿈(앞뒤 공백 포함)을 공백 하나로 바꾸세요

import re
HEADER = re.compile(
    r"""
    ^\s*
    (?:
        (?:국가별|품목별)\s+ICT\s+시장동향
        |
        [ⅠⅡⅢⅣⅤⅥⅦⅧⅨⅩ]+\s+ICT[^\n]*
    )
    \s*\n
    \s*\d{1,3}\s*
    (?:\n|$)
    """,
    re.VERBOSE,
)


def clean(text):
    text = HEADER.sub("", text)
    text = re.sub(r"\s*\n\s*", " ", text)     # 줄바꿈을 공백 하나로
    return text.strip()

In [70]:
raw = pages[5].page_content
result = clean(raw)
result

'2. ICT 정부 기관 ① 총무성 표 | 일본 총무성(MIC) 출처 : 일본 총무성 구분 내용 기관명 총무성 (総務省, Ministry of Internal Affairs and Communications) 설립 연도 2001년 주요 산하 기관 정보통신국(情報通信国), 국립정보통신기술연구소(NICT), 사이버보안총괄관(サイバーセキュリティ統括官) 등 홈페이지 www.soumu.go.jp 주요 인사 § 무라카미 세이이치로 (村上\u3000誠一郎) § 일본 총무성 총무대신 § 2024년 10월 총무대신 취임(석파내각) § 2024년 11월 연임(제2차 석파내각) 조직도 n 국가 기본 구조와 국민 생활 기반에 관한 핵심 행정 기능 수행 - 행정 운영 개선, 지방 행정, 선거, 소방 방재 등 국가 기반 행정 업무 총괄 - 통신 및 우편 서비스 등 국민 생활 관련 서비스 제공 및 관리 담당 - ICT 정책의 수립 및 실행, 사이버보안 정책 추진 n ICT 기반 구축 및 국제 협력을 통한 디지털 혁신 가속화 - 2024년 6월, 5G 및 차세대 통신 기술 발전 및 네트워크 환경 고도화를 위한 주파수 재편 계획 발표 - 2024년 8월, 사회 전반에 디지털 기술 도입을 확산시키기 위한 디지털 전환 (DX) 촉진 전략 수립. 이 전략에는 행정 서비스의 디지털화, 디지털 인프라 강화, 디지털 인재 양성 등이 포함 - 2024년 11월, 정부 및 공공기관의 보안 체계 강화를 위해 사이버 공격 대응 능력 향상, 정보 시스템 보호 조치 확대 등을 포함하는 사이버보안 강화 방안 마련'

> **주의:** "숫자만 있는 줄을 모두 지운다"로 쪽 번호를 없애면 표 안의 숫자 칸(연도, 금액)까지 지워진다. 머리말 바로 아래 줄만 지우도록 범위를 좁힌다. 정리 규칙은 문서마다 다르므로, 먼저 실제 텍스트를 출력해 보고 정한다.

### 4-3. 청킹 — split_documents는 메타데이터를 지킨다

Part 1은 `split_text`로 문자열을 잘랐다. 문서가 `Document`이면 `split_documents`를 쓴다. 잘린 조각마다 원래 쪽의 `metadata`가 그대로 복사된다.

이번에는 `from_tiktoken_encoder`로 **토큰 수** 기준으로 자른다. Part 1의 `chunk_size`는 글자 수였다.

In [71]:
# TODO: all_pages를 메타데이터가 유지되도록 잘라 pdf_chunks에 담으세요

# 원본 문서 (페이지번호 포함) -> 클린징 문서(페이지번호 삭제)
def load_clean(path):
    docs = load_pdf(path)
    for d in docs:
        d.page_content = clean(d.page_content)
    return docs

In [72]:
#모든 페이지를 클린징 문서로 바꾼다
all_pages = [d for name in PDF_FILES for d in load_clean(Path("data")/name)]

#스플리터
pdf_spitter = RecursiveCharacterTextSplitter.from_tiktoken_encoder(chunk_size=300, chunk_overlap=50)
pdf_chunk = pdf_spitter.split_documents(all_pages)

In [73]:
len(all_pages), len(pdf_chunk)

(103, 734)

### 4-4. pgvector에 저장

Part 1과 같은 `PGVector`에 `add_documents`로 넣는다. `metadata`도 함께 저장된다(`use_jsonb=True`).

> **참고:** 처음 실행하면 `Collection not found`가 한 번 출력된다. `pre_delete_collection=True`가 지울 컬렉션을 찾았는데 아직 없다는 뜻이고, 오류가 아니다.

In [75]:
store = PGVector(
    embeddings=embeddings,
    collection_name="ict_reports",
    connection=pg_engine,
    use_jsonb=True,
    pre_delete_collection=True
)

store.add_documents(pdf_chunk)

['28909f4b-c565-4695-8c1d-a3c32fa5f299',
 '782d8dc7-4361-4153-a922-03509d269f1a',
 '5f2452ba-036a-43da-bb8b-3840330e032f',
 '24697e88-aac7-4ea4-b1a3-df5ce54a8388',
 '693b7e3c-c0e7-430a-9f8b-d5579bfe515e',
 '8dff75d9-f2e0-4e56-9e4d-9bda9c5917aa',
 'a2f4940e-9dfa-4ede-a6b2-56070303832c',
 'a4909094-822d-41dd-a3d5-35b2189bf473',
 '10bb3744-94d8-41a0-b7b7-5e2852d96ae4',
 '813c853f-45ae-4c61-9e74-83f1d1113df4',
 '6c7cac5d-6581-4a68-9cb8-5b3ac02aec42',
 'c19f4143-34ec-4e8f-820b-17cab688b5b4',
 'c63a6ed4-27d3-4792-860d-b3a514292db2',
 '83eb19ae-96f1-4551-b1f9-3c0dce2e1920',
 'ef52f338-f567-4ef3-ab85-ec9b619ea290',
 '63c6b96b-8adb-4801-8807-7ab05a5fbb58',
 'da5b4d21-1923-4c64-9d1a-0d33628acc4e',
 '67272c19-abc0-450a-9553-ae694b08092a',
 '713e162f-af08-47d3-bbdc-bf5cb6bb9511',
 '697531d9-8af6-49b9-bb94-a4d4a2277f48',
 '0e7589dc-8c6d-46cf-b26f-724ffc3dfe40',
 '1a02846d-91b2-4a7a-a3ed-bca8dbabe31d',
 'fbcb4638-05f8-4962-a1a3-47e41d4c5cd5',
 '88d6e54b-d9b2-4e82-b0d3-102f9f50b78a',
 '3156cf34-d3fc-

### 4-5. 근거 쪽 번호와 함께 답하기

3-4에서는 답변과 근거를 따로 출력했다. 이번에는 근거마다 `[파일명 쪽]`을 붙여 프롬프트에 넣고, 답변 문장에도 그 표시를 붙이게 한다.

In [ ]:
# TODO: 조각마다 "[파일명 N쪽]" 한 줄과 본문을 붙이고, 조각 사이는 빈 줄로 이어 반환하세요
def format_docs_with_source(docs):
    return '\n'.join(f"[{d.metadata['source']} {d.metadata['page']}쪽]\n{d.page_content}" for d in docs)

pdf_prompt = ChatPromptTemplate.from_template(
    "다음 근거만 사용해 답한다.\n"
    "문장마다 끝에 근거의 [파일명 쪽] 표시를 그대로 붙인다.\n"
    "근거에 없는 내용이면 '문서에 없습니다'라고만 답하고 표시를 붙이지 않는다.\n\n"
    "근거:\n{context}\n\n질문: {question}"
)


In [ ]:
#래그 리트리버 생성
pdf_retriever = store.as_retriever(search_kwargs={"k":4})

#래그체인생성
pdf_chain = (
    { "context":pdf_retriever | format_docs_with_source, 
     "question": RunnablePassthrough()
     }
) | pdf_prompt | llm | StrOutputParser()

In [ ]:
# 답변의 [파일명 쪽] 표시가 실제 검색 결과와 맞는지 확인

In [83]:
#래그 질의
questions = ["일본 정부의 AI 관련 정책을 알려 줘", "글로벌 블록체인 시장 규모 전망은?", "한국의 올해 반도체 수출액은?"]
for q in questions:
    print('Q : ', q)
    print(pdf_chain.invoke(q))
    print()

Q :  일본 정부의 AI 관련 정책을 알려 줘
일본 정부는 AI 규제법을 검토하고 있으며, 기업에 법적 의무를 부과하는 방안을 고려하고 있습니다. 이는 AI 기술이 허위 정보 생성, 개인정보 침해 등의 사회적 위험을 초래할 경우 법적 규제 적용 및 위반 시 제재 방안을 논의하는 내용이 포함됩니다 [ict_japan.pdf 19쪽]. 또한 일본 AI 시장은 연평균 23.2% 성장할 것으로 전망되며, 2027년에는 1조 엔을 돌파할 것으로 보입니다 [ict_japan.pdf 19쪽].

Q :  글로벌 블록체인 시장 규모 전망은?
글로벌 블록체인 시장은 2025년 411억 달러에서 2027년 962억 달러로 성장할 전망입니다. [blockchain.pdf 3쪽]

Q :  한국의 올해 반도체 수출액은?
문서에 없습니다.



In [ ]:
# 답변의 [파일명 쪽] 표시가 실제 검색 결과와 맞는지 확인 (원문)


### 4-6. 오류 대응 — 검색 결과를 그대로 프롬프트에 넣으면?

`{"context": pdf_retriever}`처럼 `format_docs` 없이 리트리버를 바로 연결하면, 검색 결과(`Document` 리스트)가 문자열로 바뀌어 그대로 들어간다. 프롬프트에 실제로 무엇이 들어가는지 비교한다.

오류는 나지 않는다. 대신 `Document(id=…, metadata={…}, page_content='…')` 같은 파이썬 표현이 섞여 들어가 토큰을 낭비하고, 쪽 번호가 문장 속에 묻혀 모델이 출처 표시를 제대로 붙이지 못한다. 리트리버 뒤에는 항상 문자열로 바꾸는 함수를 둔다.

In [ ]:
q = '글로벌 블록체인 시장 규모 전망은?'

### 4-7. 메타데이터 필터 — 파일 하나에서만 찾기

`filter`로 `metadata` 조건을 걸면 그 조건에 맞는 조각 안에서만 검색한다. 같은 질문을 필터 없이, 그리고 일본 보고서로만 좁혀 검색한다.

In [87]:
q = "정부의 AI 정책"

for d in store.as_retriever(search_kwargs={"k": 4, "filter":{"source":"ict_japan.pdf"}}).invoke(q):
    print(" - ", d.metadata)

 -  {'page': 12, 'source': 'ict_japan.pdf'}
 -  {'page': 19, 'source': 'ict_japan.pdf'}
 -  {'page': 19, 'source': 'ict_japan.pdf'}
 -  {'page': 13, 'source': 'ict_japan.pdf'}


### 관찰

| 확인 | 결과 |
| --- | --- |
| 4-2 잡음 정리 | 머리말·쪽 번호가 사라지고 문장이 한 줄로 이어진다 |
| 4-3 `split_documents` | 조각마다 `source`·`page`가 남는다. `split_text`는 문자열만 남긴다 |
| 4-5 답변 | 문장 끝에 `[파일명 쪽]`이 붙는다. 문서에 없는 질문은 "문서에 없습니다"라고만 답한다 |
| 4-6 그대로 넣기 | 오류 없이 `Document(`·`metadata=`가 프롬프트에 섞인다 |
| 4-7 필터 | 필터가 없으면 미국 보고서 위주로 나오고, 필터를 걸면 일본 보고서에서만 나온다 |

> **참고:** Day14 보충 실습(`M4-흐름제어/Day14-보충-에이전틱RAG-실습.ipynb`)에서는 4-7의 필터를 파일별 **검색 도구**로 만들어, 질문에 맞는 보고서를 에이전트가 고르게 한다.

### Day 12(M3 Day03) 정리

**확인 질문**
- 수동 코사인 유사도(Day01)와 pgvector 리트리버(오늘)는 코치 입장에서 무엇이 달라졌고 무엇이 그대로인가?
- 이 RAG(장기 기억)에 Day02의 세션 메모리(단기 기억)를 더한 지금 구조가, M4에서 배울 Hybrid Memory와 어떻게 이어질지 예상해보자.
- PDF에서 꺼낸 조각에 파일명·쪽 번호를 남겨 두면, 답변의 근거를 사람이 원문에서 직접 확인할 수 있다. 이것이 왜 중요한가?

## 확인 문제

1. RAG 6단계 흐름을 순서대로 말하면?
2. 1-5의 오류는 무엇을 잘못 설정했을 때 생겼나? 1-6은 왜 '오류'가 아니라 '실측 확인'이었나?
3. Part 2와 Part 2의 도메인 반복에서, 작은 청크·큰 청크의 장단점이 도메인이 달라도 그대로 통했는가?
4. "근거에 없으면 모른다"는 지시가 없다면 3-3의 결과는 어떻게 달라졌을까?
5. `rag_session_coach`에서 Day01·Day02·M1 Day04·오늘 각각에서 가져온 요소는 무엇인가?
6. 4-2에서 "숫자만 있는 줄을 모두 지우기" 대신 머리말 바로 아래 줄만 지운 이유는?
7. `split_text`와 `split_documents`는 결과에서 무엇이 다른가?
8. 4-6처럼 검색 결과를 그대로 프롬프트에 넣으면 오류가 나지 않는데도 왜 문제인가?